# Phase 2 — Extended Feature Engineering & Ablation Studies

This notebook extends Section V of the baseline documentation, building directly on discoveries
from [01_deep_eda.ipynb](01_deep_eda.ipynb): aggregating all promotions into a generic `has_promo` flag
conceals genuine signals, as promotional channels and stacking rules exhibit divergent impacts
(`social_media` +8.3%, `stackable=1` +6.1%, while `in_store` and `all_channels` are negative).

Four candidate feature groups are evaluated against the 53 baseline features:

| Feature Group | Description | Leakage Prevention Rationale |
|---|---|---|
| A. Extended Promotions | `promo_channel` (one-hot), `promo_type`, `stackable_flag`, `min_order_value`, days remaining | Promotional schedules are published in advance — identical rationale as baseline Section V |
| B. Past COGS Lags | `cogs` lagged $\ge H$ days ($H=28$), rolling stats, past margin | Contemporary COGS is a leak (proved in Section V.5), but COGS shifted $\ge H$ days is strictly historical |
| C. Major Standalone Holidays | Countdown to 11/11 and 12/12 shopping festivals | Fixed calendar events known deterministically in advance |
| D. Explicit Interactions | `promo_channel × is_weekend`, `promo_any × is_weekend` | Derived strictly from valid, non-leaking antecedent features |

**Methodological Discipline:** Every historical series (such as COGS) must pass an automated corruption test.
Furthermore, **no feature is retained merely because it sounds plausible**: every candidate group must prove an
empirical reduction in cross-validation WAPE (Fold 2, Fold 3) before adoption into the final production feature set.


## 1. Environment & Setup


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from src.config import (
    DATA_RAW, DATA_PROCESSED, RESULTS, H, TARGET, SEED,
    FOLDS, TEST_FOLD, CV_FOLDS, TET_DATES, RETAIL_SPECIAL_DAYS,
    BASE_GROUP_PREFIXES,
)
from src.data import load_clean_sales, load_promotions, split_by_date
from src.metrics import evaluate_metrics
from src.models import fit_lightgbm, evaluate_fold, cross_validate
from src.features import (
    add_lag_features, add_calendar_features, add_tet_features,
    add_promo_features, add_promo_features_v2, add_cogs_lag_features,
    add_specific_holiday_features, leakage_probe_cogs,
)

raw = load_clean_sales()
promotions = load_promotions()
print(f"sales_clean: {len(raw):,} rows   promotions: {len(promotions)} campaigns")
print("promo_type :", sorted(promotions.promo_type.dropna().unique()))
print("promo_channel:", sorted(promotions.promo_channel.dropna().unique()))


sales_clean: 3,833 dòng   promotions: 50 chương trình
promo_type : ['fixed', 'percentage']
promo_channel: ['all_channels', 'email', 'in_store', 'online', 'social_media']


## 2. Reconstructing Baseline Features (53 Columns)

We reconstruct the four baseline feature transforms to establish a benchmark.
All new feature groups below are evaluated strictly on their marginal contribution relative to this 53-feature baseline.


In [2]:
base = add_promo_features(
    add_tet_features(add_calendar_features(add_lag_features(raw.copy()))),
    promotions,
)
base_features = [c for c in base.columns if c not in ("date", TARGET, "cogs")]
print(f"Base feature set: {len(base_features)} columns (baseline document: 53 columns)")
assert len(base_features) == 53, "Feature count mismatch against baseline specification"


Bộ đặc trưng gốc: 53 cột (tài liệu gốc: 53 cột)


## 3. Candidate Group A — Extended Promotions

Baseline promotional processing only extracted counts, maximum discount, and broad category flags,
omitting channel distribution, stacking rules, and threshold constraints.
When multiple promotions overlap on a single date, categorical attributes resolve to the campaign
with the largest `discount_value`, while individual channels are marked independently.


In [3]:
CHANNELS = sorted(promotions.promo_channel.dropna().unique())
ext = add_promo_features_v2(base.copy(), promotions)
new_promo_cols = ["promo_is_percentage", "stackable_flag", "min_order_value",
                   "promo_days_left"] + [f"promo_ch_{c}" for c in CHANNELS]
print(f"Added {len(new_promo_cols)} columns: {new_promo_cols}")
ext[["date"] + new_promo_cols].loc[ext.promo_any == 1].head(5)


Thêm 9 cột: ['promo_is_percentage', 'stackable_flag', 'min_order_value', 'promo_days_left', 'promo_ch_all_channels', 'promo_ch_email', 'promo_ch_in_store', 'promo_ch_online', 'promo_ch_social_media']


,date,promo_is_percentage,stackable_flag,min_order_value,promo_days_left,promo_ch_all_channels,promo_ch_email,promo_ch_in_store,promo_ch_online,promo_ch_social_media
211,2013-01-31,1,0,0.0,29,0,0,1,0,0
212,2013-02-01,1,0,0.0,28,0,0,1,0,0
213,2013-02-02,1,0,0.0,27,0,0,1,0,0
214,2013-02-03,1,0,0.0,26,0,0,1,0,0
215,2013-02-04,1,0,0.0,25,0,0,1,0,0


## 4. Candidate Group B — Cost of Goods Sold (COGS) Lags

Contemporary (same-day) COGS represents fatal leakage ($r = 0.976$ with same-day revenue, only observable post-sale).
However, COGS lagged $\ge H$ days ($H=28$) is strictly historical and satisfies time-series feasibility.
We engineer historical COGS lags and a lagged gross margin indicator (`margin_lag_28`).


In [4]:
ext = add_cogs_lag_features(ext)
new_cogs_cols = ["cogs_lag_28", "cogs_lag_91", "cogs_lag_364",
                  "cogs_roll_mean_28", "cogs_roll_mean_91", "margin_lag_28"]
print(f"Added {len(new_cogs_cols)} columns: {new_cogs_cols}")
ext[["date"] + new_cogs_cols].tail(3)


Thêm 6 cột: ['cogs_lag_28', 'cogs_lag_91', 'cogs_lag_364', 'cogs_roll_mean_28', 'cogs_roll_mean_91', 'margin_lag_28']


,date,cogs_lag_28,cogs_lag_91,cogs_lag_364,cogs_roll_mean_28,cogs_roll_mean_91,margin_lag_28
3830,2022-12-29,1683082.90,4023586.93,3948015.94,1.576647e+06,2.021873e+06,-0.008773
3831,2022-12-30,726739.78,2332362.64,3121467.33,1.570159e+06,2.010664e+06,-0.008827
3832,2022-12-31,751384.00,1293984.10,3019583.50,1.540113e+06,2.004420e+06,0.003078


## 5. Candidate Group C — Major Shopping Festivals (11/11 & 12/12)

The baseline binary flag `is_special_day` lumped 10 retail dates together, treating mega shopping festivals
(Single's Day 11/11 and Double Twelve 12/12) identically to minor holidays. Here we engineer targeted countdown distance metrics.


In [5]:
check = base.copy()
check["year_ratio"] = check[TARGET] / check.groupby(check.date.dt.year)[TARGET].transform("mean")
is_1111 = (check.date.dt.month == 11) & (check.date.dt.day == 11)
is_1212 = (check.date.dt.month == 12) & (check.date.dt.day == 12)
is_other_special = (check.is_special_day == 1) & ~is_1111 & ~is_1212

print(f"11/11 Single's Day   : {check.loc[is_1111, 'year_ratio'].mean():.3f}  ({is_1111.sum()} days)")
print(f"12/12 Double Twelve  : {check.loc[is_1212, 'year_ratio'].mean():.3f}  ({is_1212.sum()} days)")
print(f"Other special days   : {check.loc[is_other_special, 'year_ratio'].mean():.3f}  ({is_other_special.sum()} days)")
print(f"Regular days         : {check.loc[check.is_special_day == 0, 'year_ratio'].mean():.3f}")


11/11            : 0.632  (11 ngày)
12/12            : 0.534  (11 ngày)
Ngày đặc biệt khác: 1.105  (84 ngày)
Ngày thường       : 1.000


In [6]:
ext = add_specific_holiday_features(ext)
new_holiday_cols = ["days_to_1111", "days_to_1212"]
print(f"Added {len(new_holiday_cols)} columns: {new_holiday_cols}")


Thêm 2 cột: ['days_to_1111', 'days_to_1212']


## 6. Candidate Group D — Explicit Interaction Terms

While gradient boosted trees can capture non-linear interactions across split levels,
explicitly materializing domain interactions (such as social media marketing coinciding with weekends)
reduces the sample complexity required for tree partitions in sparse regimes.


In [7]:
ext["promo_weekend"] = ext.promo_any * ext.is_weekend
ext["promo_social_weekend"] = ext.get("promo_ch_social_media", 0) * ext.is_weekend
new_interaction_cols = ["promo_weekend", "promo_social_weekend"]
print(f"Added {len(new_interaction_cols)} columns: {new_interaction_cols}")


Thêm 2 cột: ['promo_weekend', 'promo_social_weekend']


## 7. Automated Data Leakage Verification

Candidate Group B (COGS lags) is derived from historical sequence values.
We execute a corruption stress test: perturbing both revenue and COGS by 930% inside the forbidden horizon boundary ($t - H < s \le t$).
Any observable variation in the generated feature values signals illegal lookahead leakage.


In [8]:
worst, n_checkpoints, n_columns = leakage_probe_cogs(raw)
print(f"Corrupted most recent {H} labels (revenue + cogs) at {n_checkpoints} checkpoints across {n_columns} columns")
print(f"Maximum observed feature delta = {worst:.3e}")
assert worst < 1e-9, "LEAKAGE DETECTED: Features observed forbidden labels inside the forecast horizon"
print("PASSED — Strict leakage avoidance verified.")


Phá hoại 28 nhãn gần nhất (revenue + cogs) tại 8 mốc, kiểm 30 cột
Thay đổi lớn nhất quan sát được = 0.000e+00
ĐẠT — không rò rỉ.


## 8. Ablation Study: Does Each Group Improve Cross-Validation WAPE?

Using an identical LightGBM regressor configuration and identical temporal fold boundaries,
we incrementally ablate candidate feature groups. Decisions are governed strictly by cross-validation WAPE
(mean of Fold 2 and Fold 3); test scores are isolated and non-decisional.


In [10]:
ready = ext.dropna(subset=["lag_371", "roll_mean_364", "cogs_lag_364"]).reset_index(drop=True)
print(f"After trimming initialization warm-up period: {len(ready):,} rows"
      f", from {ready.date.min().date()} to {ready.date.max().date()}")

FEATURE_SETS = {
    "A. Baseline (53 cols)": base_features,
    "B. + Extended Promotions": base_features + new_promo_cols,
    "C. + Lag COGS": base_features + new_promo_cols + new_cogs_cols,
    "D. + Standalone Holidays": base_features + new_promo_cols + new_cogs_cols + new_holiday_cols,
    "E. + Interactions (Full)": (base_features + new_promo_cols + new_cogs_cols
                                 + new_holiday_cols + new_interaction_cols),
}

rows = []
for name, columns in FEATURE_SETS.items():
    scores = cross_validate(ready, columns)
    train, valid = split_by_date(ready, *TEST_FOLD[1:])
    model = fit_lightgbm(train, columns, seed=SEED)
    pred = np.expm1(model.predict(valid[columns]))
    scores["TEST"] = evaluate_metrics(valid[TARGET].values, pred)["WAPE"]
    rows.append({"set": name, "n_columns": len(columns), **scores})

table = pd.DataFrame(rows).set_index("set")
base_cv = table.loc["A. Baseline (53 cols)", "CV"]
table["vs_A_pct"] = 100 * (table.CV / base_cv - 1)
print(table[["n_columns", "Fold 1", "Fold 2", "Fold 3", "CV", "vs_A_pct", "TEST"]]
      .round(4).rename_axis("Feature Candidate Set").to_string())


Sau khi bỏ giai đoạn khởi động (cần cả lag_371 và cogs_lag_364): 3,462 dòng, từ 2013-07-10 đến 2022-12-31


                         n_columns  Fold 1  Fold 2  Fold 3      CV  vs_A_pct    TEST
bộ đặc trưng                                                                        
A. gốc (53 cột)                 53  0.4904  0.2628  0.2172  0.2400    0.0000  0.2072
B. + khuyến mãi mở rộng         62  0.5156  0.2602  0.2181  0.2391   -0.3529  0.2008
C. + lag COGS                   68  0.4824  0.2561  0.2302  0.2431    1.3179  0.2069
D. + ngày lễ riêng biệt         70  0.4796  0.2673  0.2276  0.2474    3.0984  0.1991
E. + tương tác (đầy đủ)         72  0.4560  0.2604  0.2323  0.2464    2.6610  0.2022


**Interpreting the Ablation Table:** A negative `vs_A_pct` indicates a genuine reduction in CV WAPE relative to baseline.
Only Feature Set B (+ Extended Promotions) demonstrated genuine generalization gain.
COGS lags, holiday countdowns, and explicit interaction terms degraded CV performance due to noise overfitting, and were pruned.


## 9. Finalizing Feature Matrix for Phase 3


In [11]:
best_set_name = table["CV"].idxmin()
final_columns = FEATURE_SETS[best_set_name]
print(f"Optimal feature candidate by CV: {best_set_name}  ({len(final_columns)} columns)")
print(f"CV WAPE = {table.loc[best_set_name, 'CV']:.4f}"
      f"  vs. baseline {base_cv:.4f}  ({table.loc[best_set_name, 'vs_A_pct']:+.2f}%)")

final_groups = {}
for name, prefixes in BASE_GROUP_PREFIXES.items():
    cols = [c for c in base_features if c.startswith(prefixes)]
    final_groups[name] = {"n_columns": len(cols), "columns": cols}

candidate_new_groups = {"khuyến mãi mở rộng": new_promo_cols, "lag COGS": new_cogs_cols,
                         "ngày lễ riêng biệt": new_holiday_cols, "tương tác": new_interaction_cols}
for name, cols in candidate_new_groups.items():
    kept = [c for c in cols if c in final_columns]
    if kept:
        final_groups[name] = {"n_columns": len(kept), "columns": kept}

out_path = DATA_PROCESSED / f"features_h{H}_v2.csv"
ready[["date", TARGET, "cogs"] + final_columns].to_csv(out_path, index=False)
print(f"Saved {out_path}  ({len(ready):,} rows, {len(final_columns)} features)")

import json
summary = {
    "ablation_table": table.round(6).to_dict("index"),
    "chosen_set": best_set_name,
    "final_features": final_columns,
    "n_final_features": len(final_columns),
    "new_groups": {"khuyến mãi mở rộng": new_promo_cols, "lag COGS": new_cogs_cols,
                   "ngày lễ riêng biệt": new_holiday_cols, "tương tác": new_interaction_cols},
}
(RESULTS / "02_features.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, default=lambda o: o.tolist()),
    encoding="utf-8")
print("Saved results/02_features.json")


Bộ đặc trưng tốt nhất theo CV: B. + khuyến mãi mở rộng  (62 cột)
CV WAPE = 0.2391  so với gốc 0.2400  (-0.35%)


Đã ghi D:\Document\aio2026\module03\conquer\data\processed\features_h28_v2.csv  (3,462 dòng, 62 đặc trưng)
Đã ghi results/02_features.json


**Phase 2 Synthesis:** The selected 62-feature dataset is saved to `data/processed/features_h28_v2.csv`
and ablation metrics are recorded in `results/02_features.json`.
Subsequent modeling phases consume this standardized matrix directly.
